## 4. Parse Player Information

Let's extract player data from the identified tbody elements:

# function

In [4]:
def fetch_player_ALL(url):
    import requests
    import re
    from bs4 import BeautifulSoup
    import json
    import pandas as pd

    response = requests.get(url)
    html_doc = response.text

    soup = BeautifulSoup(html_doc, 'html.parser')
    games_data = []

    for script in soup.find_all('script'):
        if script.string and "var games = [" in script.string:
            match = re.search(r"var games = \[.*?\];", script.string, re.DOTALL)
            if match:
                games_str = match.group(0).replace('var games =', '').strip(' ;')
                try:
                    games_data = json.loads(games_str)
                    break
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")

    # Extract subfields from 'GameTeam' if present
    for game in games_data:
        if 'GameTeam' in game and isinstance(game['GameTeam'], dict):
            game['GameTeam_TEAM'] = game['GameTeam'].get('Name', {})
            game['GameTeam_GameResult'] = game['GameTeam'].get('Game', {}).get('Result')
            game['GameTeam_GameGuid'] = game['GameTeam'].get('Game', {}).get('Guid')
            game['GameTeam_GameDate'] = game['GameTeam'].get('Game', {}).get('Date')
            game['GameTeam_AwayTeamName'] = game['GameTeam'].get('Game', {}).get('AwayTeam', {}).get('Name')
            game['GameTeam_HomeTeamName'] = game['GameTeam'].get('Game', {}).get('HomeTeam', {}).get('Name')

            # Add AGE column by trimming after the 2nd last space in GameTeam_TEAM
            for game in games_data:
                team_name = game.get('GameTeam_TEAM', '')
                if isinstance(team_name, str):
                    parts = team_name.split(' ')
                    if len(parts) > 2:
                        game['AGE'] = ' '.join(parts[-2:])
                    else:
                        game['AGE'] = team_name
                else:
                    game['AGE'] = None
    

    # filtered_games_data = [
    #     {k: game[k] for k in fields_to_keep if k in game}
    #     for game in games_data
    # ]

    df_games = pd.DataFrame(games_data)
    
    df_games = df_games.drop(columns=['id', 'GameTeamId', 'Stints', 'createdAt', 'updatedAt', 'GameTeam'])
    return df_games


In [3]:
import re


def extract_player_data(tbody, team_type):
    """Extract player information from a tbody element"""
    players = []
    staff = []
    
    # Find rows with player data
    player_rows = tbody.find_all('tr', class_='ng-scope')
    
    for row in player_rows:
        # Get all td elements in the row
        cells = row.find_all('td')
        
        # Skip rows with insufficient data
        if len(cells) < 3:
            continue
            
        # Try to determine if this is a player or staff member
        is_staff = False
        role = ""
        
        # Check for coach, assistant coach, or delegate indicators
        for cell in cells:
            text = cell.get_text().strip()
            if text in ["Coach", "Ass. Coach", "Gedelegeerde"]:
                is_staff = True
                role = text
                break
        
        # Extract player/staff information based on the row structure
        if is_staff:
            # Staff member (coach, assistant, delegate)
            name = ""
            dob = ""
            
            # Extract name and DOB from appropriate cells based on their position
            for idx, cell in enumerate(cells):
                text = cell.get_text().strip()
                if idx >= 3 and text and "button" not in str(cell) and role not in text:
                    if re.search(r'\d{1,2}-\d{1,2}-\d{4}', text):
                        dob = text.split('(')[0].strip()  # Extract DOB part
                    elif text:
                        name = text
            
            if name:  # Only add if we found a name
                staff.append({
                    "team": team_type,
                    "role": role,
                    "name": name,
                    "dob": dob
                })
                
        else:
            # Player
            jersey_number = ""
            player_name = ""
            category = ""
            
            # Extract jersey number from SVG text or cell content
            number_cell = None
            for cell in cells:
                if cell.find('svg', class_=lambda c: c and 'shirt_color_icon' in c):
                    number_cell = cell
                    # Try to extract number from text content in the SVG
                    svg_text = cell.find('text')
                    if svg_text:
                        jersey_number = svg_text.get_text().strip()
                    break
            
            # If no jersey number found in SVG, look for it in cell text
            if not jersey_number and number_cell:
                jersey_number = number_cell.get_text().strip()
                
            # Find player name and category by position or content
            for cell in cells:
                text = cell.get_text().strip()
                
                # Category usually has format like " (J16)"
                if "(" in text and ")" in text and len(text) < 10:
                    category = text.strip()
                    
                # Player name is typically in a cell without special elements
                elif text and not cell.find('svg') and not cell.find('button'):
                    # Skip cells with special content
                    if text not in ["N", "Y", "Cap"] and not re.match(r'^\d+$', text):
                        player_name = text
            
            # Only add if we found a name
            if player_name:
                # Check if this player is the captain
                is_captain = bool(row.select('i:contains("Cap"):not(.ng-hide)'))
                
                players.append({
                    "team": team_type,
                    "jersey_number": jersey_number.strip(),
                    "name": player_name,
                    "category": category,
                    "captain": is_captain
                })
    
    return players, staff

# Process each tbody with player data
all_players = []
all_staff = []

for i, tbody in enumerate(player_tbodies):
    # Determine if this is home or away team data
    team_type = "Unknown"
    if "deelnemer in Tthuis" in str(tbody):
        team_type = "Home"
    elif "deelnemer in Tuit" in str(tbody):
        team_type = "Away"
        
    players, staff = extract_player_data(tbody, team_type)
    all_players.extend(players)
    all_staff.extend(staff)
    
    print(f"Extracted {len(players)} players and {len(staff)} staff members from tbody #{i+1} ({team_type} team)")

# Remove players where jersey_number is null or empty
all_players = [p for p in all_players if p.get("jersey_number")]

# Create DataFrames from the extracted data
players_df = pd.DataFrame(all_players)
def get_players_df():
    return players_df

NameError: name 'player_tbodies' is not defined

In [5]:
def FETCH_PLAYERS_AVG(url):
    df = fetch_player_ALL(url)
    # Unnest AGE from GameTeam column if not already present
    if 'AGE' not in df.columns:
        df['AGE'] = df['GameTeam'].apply(lambda x: ' '.join(x['Name'].split(' ')[-2:]) if isinstance(x, dict) and 'Name' in x else None)

    grouped = df.groupby(['Name', 'AGE'])
    result = grouped.agg({
        'TotalMinutes': 'mean',
        'NormalizedMinutes': 'mean',
        'FreeThrows': 'mean',
        'FieldGoals': 'mean',
        'ThreePointers': 'mean',
        'TotalScore': ['min', 'mean', 'median', 'max' , 'count'],
        'PlusMinus': 'mean',
        'Faults': 'mean',
        'Plus': 'mean',
        'Minus': 'mean',
       
    })

    # Flatten MultiIndex columns
    result.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in result.columns.values]
    # Rename and rearrange columns to match the desired order and names
    result = result.rename(columns={
         'TotalScore_count': 'WEDSTRIJDEN',
        
        'TotalMinutes_mean': 'Avg TotalMinutes',
        'NormalizedMinutes_mean': 'Avg NormalizedMinutes',
        'FreeThrows_mean': 'Avg FreeThrows',
        'FieldGoals_mean': 'Avg FieldGoals',
        'ThreePointers_mean': 'Avg ThreePointers',
        'TotalScore_min': 'Min TotalScore',
        'TotalScore_mean': 'Avg TotalScore',
        'TotalScore_median': 'Median TotalScore',
        'TotalScore_max': 'Max TotalScore',
       'Faults_mean': 'Avg Faults',
        'PlusMinus_mean': 'Avg PlusMinus',
        
        'Plus_mean': 'Avg Plus',
        'Minus_mean': 'Avg Minus'
    })

    # Reorder columns to match the desired template
    desired_order = [
        'WEDSTRIJDEN',
        'Avg TotalMinutes',
        'Avg NormalizedMinutes',
        'Avg FreeThrows',
        'Avg FieldGoals',
        'Avg ThreePointers',
        'Min TotalScore',
        'Avg TotalScore',
        'Median TotalScore',
        'Max TotalScore',
        
        'Avg PlusMinus',
        'Avg Faults',
        'Avg Plus',
        'Avg Minus'
    ]
    # Keep index columns (Name, AGE) at the front
    result = result.reset_index()[['Name', 'AGE'] + desired_order]
    # Round all columns containing 'Avg' to 1 decimal place
    avg_cols = [col for col in result.columns if 'Avg' in col]
    result[avg_cols] = result[avg_cols].round(1)
    return result




In [ ]:
with open(r'C:\Users\StijnHuysman\OneDrive - mateco cloud\GITHUB REPOS\HAANTJES\ALLPLAYERS.json', 'r', encoding='utf-8') as f:
    allplayers_data = json.load(f)
# Convert allplayers_data to a DataFrame and ensure required columns exist
allplayers_df = pd.DataFrame(allplayers_data) 
# If 'rows' column contains dictionaries, expand them into separate columns
if isinstance(allplayers_df['rows'].iloc[0], dict):
    rows_expanded = allplayers_df['rows'].apply(pd.Series)
    allplayers_df = pd.concat([allplayers_df.drop(columns=['rows']), rows_expanded], axis=1)
    display(allplayers_df.head(1))
else:
    display(allplayers_df['rows'].head(1))

,total,totalNotFiltered,Guid,LidNr,Name,createdAt,updatedAt
0,64465,64465,BVBL769698,769698,Aadhav Katthekasu,2025-06-02T18:21:38.974Z,2025-06-02T18:21:38.974Z


In [ ]:
PLAYERSBBVL = allplayers_df[['Guid', 'LidNr', 'Name']]

In [ ]:
# Merge players_df with PLAYERSBBVL on the 'name' column to get the Guid
players_df = players_df.merge(PLAYERSBBVL, left_on='name', right_on='Name', how='left')
players_df['Guid'] = players_df['Guid']  # Add Guid column from PLAYERSBBVL

# Optionally, drop the extra 'Name' column from PLAYERSBBVL
players_df = players_df.drop(columns=['Name'])
players_df_merged = players_df

In [ ]:
def merge_player_averages(players_df):
    merged_list = []
    for _, row in players_df.iterrows():
        guid = row['Guid']
        if pd.notnull(guid):
            url = f"https://app.basketballstatsvlaanderen.be/players/{guid}"
            avg_df = FETCH_PLAYERS_AVG(url)
            # Find the matching player by name and AGE
            match = avg_df[(avg_df['Name'] == row['name']) & (avg_df['AGE'] == row.get('category', None))]
            if not match.empty:
                merged_row = {**row.to_dict(), **match.iloc[0].to_dict()}
            else:
                merged_row = row.to_dict()
            merged_list.append(merged_row)
        else:
            merged_list.append(row.to_dict())
    return pd.DataFrame(merged_list)



players_with_averages_df = merge_player_averages(players_df_merged)
players_with_averages_df.head()

NameError: name 'players_df_merged' is not defined

In [ ]:
merged_rows = []
for _, row in players_df_merged.iterrows():
    guid = row['Guid']
    if pd.notnull(guid):
        url = f"https://app.basketballstatsvlaanderen.be/players/{guid}"
        avg_df = FETCH_PLAYERS_AVG(url)
        match = avg_df[avg_df['Name'] == row['name']]
        if not match.empty:
            merged_row = {**row.to_dict(), **match.iloc[0].to_dict()}
        else:
            merged_row = row.to_dict()
        merged_rows.append(merged_row)
    else:
        merged_rows.append(row.to_dict())

players_with_averages_df = pd.DataFrame(merged_rows)

In [ ]:
players_with_averages_df

,team,jersey_number,name,category,captain,Guid,LidNr,Name,AGE,WEDSTRIJDEN,...,Avg FieldGoals,Avg ThreePointers,Min TotalScore,Avg TotalScore,Median TotalScore,Max TotalScore,Avg PlusMinus,Avg Faults,Avg Plus,Avg Minus
0,Home,4,Jonah Cattoir,(J16),False,BVBL762639,762639,Jonah Cattoir,J16 A,3,...,3.3,0.0,2,4.0,4.0,6,-29.0,1.0,34.7,-63.7
1,Home,5,Viktor L'Hommelet,(J16),False,BVBL714059,714059,Viktor L'Hommelet,J16 A,3,...,1.3,2.0,0,3.7,4.0,7,-30.3,3.0,40.3,-70.7
2,Home,6,Niels De Coussemaker,(J16),False,BVBL708695,708695,Niels De Coussemaker,J16 A,3,...,1.3,0.0,1,1.7,2.0,2,-15.0,1.7,20.7,-35.7
3,Home,7,Tibo Despriet,(J16),False,BVBL738959,738959,Tibo Despriet,J16 A,3,...,7.3,2.0,10,12.0,11.0,15,-24.7,2.7,39.3,-64.0
4,Home,8,Vic Huysman,(J16),True,BVBL744354,744354,Vic Huysman,J16 A,3,...,18.7,1.0,24,25.0,25.0,26,-29.7,2.7,49.7,-79.3
5,Home,10,Gust Ottevaere,(J16),False,BVBL750587,750587,Gust Ottevaere,J16 A,3,...,0.0,0.0,0,0.0,0.0,0,-17.0,1.3,12.7,-29.7
6,Home,11,Jasper Espeel,(J16),False,BVBL749137,749137,Jasper Espeel,J16 A,3,...,0.7,0.0,1,2.0,2.0,3,-21.0,1.3,34.7,-55.7
7,Home,12,Marcel Heerman,(J16),False,BVBL668586,668593,Marcel Heerman,J16 A,1,...,12.0,0.0,13,13.0,13.0,13,-9.0,2.0,37.0,-46.0
8,Home,15,Torre Beeckman,(J16),False,BVBL720212,720212,Torre Beeckman,J16 A,3,...,0.0,1.0,0,1.7,2.0,3,-21.0,1.0,27.3,-48.3
9,Away,4,Richard Baert,(J16),False,BVBL704974,704974,Richard Baert,J16 A,3,...,4.0,0.0,2,4.7,5.0,7,-23.0,0.7,20.0,-43.0


In [ ]:
tegenploeg = players_with_averages_df[players_with_averages_df['team'] == 'Away'].sort_values('Avg TotalScore', ascending=False)
tegenploeg.to_excel('tegenploeg.xlsx', index=False)
tegenploeg

,team,jersey_number,name,category,captain,Guid,LidNr,Name,AGE,WEDSTRIJDEN,...,Avg FieldGoals,Avg ThreePointers,Min TotalScore,Avg TotalScore,Median TotalScore,Max TotalScore,Avg PlusMinus,Avg Faults,Avg Plus,Avg Minus
14,Away,9,Ewoud Vindevogel,(J16),False,BVBL718280,718280,Ewoud Vindevogel,J16 A,3,...,6.0,3.0,9,10.3,10.0,12,-33.3,2.7,33.0,-66.3
13,Away,8,Arthur Van De Walle,(J16),False,BVBL735976,735976,Arthur Van De Walle,J16 A,3,...,8.0,1.0,4,9.0,8.0,15,-36.7,2.7,34.3,-71.0
16,Away,11,Jolan Allegaert,(J16),True,BVBL735447,735447,Jolan Allegaert,J16 A,3,...,5.3,0.0,4,7.3,7.0,11,-26.3,1.3,22.0,-48.3
15,Away,10,Obe Van der Heggen,(J16),False,BVBL717962,717962,Obe Van der Heggen,J16 A,3,...,5.3,0.0,4,6.7,4.0,12,-33.0,1.7,36.0,-69.0
9,Away,4,Richard Baert,(J16),False,BVBL704974,704974,Richard Baert,J16 A,3,...,4.0,0.0,2,4.7,5.0,7,-23.0,0.7,20.0,-43.0
11,Away,6,Kas Lecluyse,(J16),False,BVBL726283,726283,Kas Lecluyse,J16 A,3,...,4.0,0.0,2,4.3,5.0,6,-26.3,1.7,20.0,-46.3
20,Away,15,Leon Calleeuw,(J16),False,BVBL718767,718767,Leon Calleeuw,J16 A,3,...,2.7,0.0,0,4.0,6.0,6,-12.7,0.0,26.7,-39.3
17,Away,12,Arthur Loyson,(J16),False,BVBL713764,713764,Arthur Loyson,J16 A,3,...,1.3,0.0,0,1.3,2.0,2,-32.0,1.3,31.7,-63.7
19,Away,14,Pepijn Raes,(J16),False,BVBL719469,719469,Pepijn Raes,J16 A,3,...,1.3,0.0,0,1.3,0.0,4,-13.0,0.7,12.7,-25.7
10,Away,5,Isaac Vandierendonck,(J16),False,BVBL723348,723348,Isaac Vandierendonck,J16 A,2,...,0.0,0.0,0,0.0,0.0,0,-13.5,0.5,7.0,-20.5


## 6. Export Data to CSV

Finally, let's export the extracted player and staff information to CSV files for future use: